# Avito Text Orientation Classification

## 1. Постановка задачи

Цель — определить вероятность того, что текстовый crop находится
вверх ногами относительно исходной ориентации.

Модель должна предсказывать `p_180` для каждого изображения.

Метрика соревнования:

`1 - Brier Score`

In [74]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet18_Weights

from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_DIR = "/content/images"

TARGET_W = 224
TARGET_H = 64

BATCH_SIZE = 64
EPOCHS = 8
LR = 1e-4
WEIGHT_DECAY = 1e-4

Device: cuda


In [75]:
from pathlib import Path

files = sorted([
    str(p)
    for p in Path(DATA_DIR).glob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
])

print("Images:", len(files))

train_files, val_files = train_test_split(
    files,
    test_size=0.2,
    random_state=SEED
)

print("Train originals:", len(train_files))
print("Val originals:", len(val_files))

Images: 20000
Train originals: 16000
Val originals: 4000


In [76]:
def resize_and_pad(image, target_w=224, target_h=64):
    image = image.convert("RGB")

    w, h = image.size

    scale = min(target_w / w, target_h / h)

    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    image = image.resize(
        (new_w, new_h),
        Image.Resampling.LANCZOS
    )

    canvas = Image.new(
        "RGB",
        (target_w, target_h),
        (255, 255, 255)
    )

    x = (target_w - new_w) // 2
    y = (target_h - new_h) // 2

    canvas.paste(image, (x, y))

    return canvas

def pil_to_tensor(image):
    image = resize_and_pad(image)

    arr = np.asarray(image).astype(np.float32) / 255.0

    tensor = torch.from_numpy(arr).permute(2, 0, 1)

    tensor = (tensor - 0.5) / 0.5

    return tensor

## 4. Synthetic training pairs

В исходном датасете отсутствуют labels, поэтому для обучения создаются
синтетические пары.

Для каждого исходного изображения `x` строится его поворот на 180°:

- `(x, rot180(x))` → label `0`
- `(rot180(x), x)` → label `1`

Таким образом, модель учится определять, какая из двух версий находится
в перевёрнутой ориентации.

Критически важно, что train/validation split выполняется на уровне
исходных изображений **до создания rotated versions**. Поэтому исходное
изображение и его 180°-версия никогда не оказываются в разных выборках.

In [ ]:
class PairRotationDataset(Dataset):

    def __init__(self, files):  
        self.files = files

    def __len__(self):
        return len(self.files) * 2

    def __getitem__(self, idx):

        file_idx = idx // 2
        reverse = idx % 2

        image = Image.open(self.files[file_idx]).convert("RGB")

        rotated = image.rotate(
            180,
            resample=Image.Resampling.BICUBIC
        )

        image_tensor = pil_to_tensor(image)
        rotated_tensor = pil_to_tensor(rotated)

        if reverse == 0:
            # first = normal
            x1 = image_tensor
            x2 = rotated_tensor
            label = 0.0

        else:
            # first = rotated
            x1 = rotated_tensor
            x2 = image_tensor
            label = 1.0

        return (
            x1,
            x2,
            torch.tensor(label, dtype=torch.float32)
        )
train_ds = PairRotationDataset(train_files)
val_ds = PairRotationDataset(val_files)

print("Train pairs:", len(train_ds))
print("Val pairs:", len(val_ds))

Train pairs: 32000
Val pairs: 8000


In [79]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [83]:
class SiameseResNet18(nn.Module):

    def __init__(self):
        super().__init__()

        backbone = models.resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        feature_dim = backbone.fc.in_features

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.head = nn.Sequential(
            nn.Linear(feature_dim * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 128),
            nn.ReLU(),

            nn.Linear(128, 1)
        )

    def forward(self, x1, x2):

        f1 = self.backbone(x1)
        f2 = self.backbone(x2)

        diff = f1 - f2
        abs_diff = torch.abs(diff)

        features = torch.cat(
            [f1, f2, diff, abs_diff],
            dim=1
        )

        logits = self.head(features)

        return logits.squeeze(1)

In [84]:
model = SiameseResNet18().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

Trainable parameters: 12291393


In [85]:
def evaluate(model, loader):

    model.eval()

    all_probs = []
    all_labels = []

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():

        for x1, x2, labels in loader:

            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(x1, x2)

            loss = criterion(logits, labels)

            probs = torch.sigmoid(logits)

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_count += batch_size

            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()

    loss = total_loss / total_count

    brier = np.mean((probs - labels) ** 2)
    score = 1.0 - brier

    accuracy = np.mean(
        (probs >= 0.5) == labels
    )

    return {
        "loss": loss,
        "brier": brier,
        "score": score,
        "accuracy": accuracy,
        "prob_mean": probs.mean(),
        "prob_std": probs.std()
    }

best_brier = float("inf")

for epoch in range(1, EPOCHS + 1):

    model.train()

    running_loss = 0.0
    count = 0

    for x1, x2, labels in train_loader:

        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(x1, x2)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        count += batch_size

    train_loss = running_loss / count

    metrics = evaluate(model, val_loader)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"brier={metrics['brier']:.4f} | "
        f"score={metrics['score']:.4f} | "
        f"accuracy={metrics['accuracy']:.4f} | "
        f"prob_std={metrics['prob_std']:.4f}"
    )

    if metrics["brier"] < best_brier:

        best_brier = metrics["brier"]

        torch.save(
            model.state_dict(),
            "/content/best_pairwise_resnet18.pt"
        )

        print(" -> saved best model")

Epoch 1/8 | train_loss=0.6835 | brier=0.2633 | score=0.7367 | accuracy=0.5034 | prob_std=0.1225
 -> saved best model
Epoch 2/8 | train_loss=0.5274 | brier=0.3317 | score=0.6683 | accuracy=0.5174 | prob_std=0.2995
Epoch 3/8 | train_loss=0.2825 | brier=0.3727 | score=0.6273 | accuracy=0.5111 | prob_std=0.3589
Epoch 4/8 | train_loss=0.1757 | brier=0.3841 | score=0.6159 | accuracy=0.5312 | prob_std=0.3958
Epoch 5/8 | train_loss=0.1198 | brier=0.3970 | score=0.6030 | accuracy=0.5200 | prob_std=0.4037
Epoch 6/8 | train_loss=0.0893 | brier=0.4090 | score=0.5910 | accuracy=0.5169 | prob_std=0.4210
Epoch 7/8 | train_loss=0.0757 | brier=0.4130 | score=0.5870 | accuracy=0.5244 | prob_std=0.4273
Epoch 8/8 | train_loss=0.0672 | brier=0.4159 | score=0.5841 | accuracy=0.5178 | prob_std=0.4264


## 7. Model selection

Для выбора checkpoint используется Brier Score на validation set.

Меньший Brier Score соответствует лучшему качеству вероятностных
предсказаний.

Validation labels являются синтетическими и используются только для
контроля обучения и выбора checkpoint. Они не являются исходной
разметкой скрытого тестового набора.

In [86]:
model.load_state_dict(
    torch.load(
        "/content/best_pairwise_resnet18.pt",
        map_location=device
    )
)

model.eval()

print("Best model loaded.")

Best model loaded.


In [87]:
class TestDataset(Dataset):

    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        path = self.files[idx]

        image = Image.open(path).convert("RGB")

        rotated = image.rotate(
            180,
            resample=Image.Resampling.BICUBIC
        )

        x = pil_to_tensor(image)
        r = pil_to_tensor(rotated)

        image_id = Path(path).stem

        return x, r, image_id

test_ds = TestDataset(files)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [88]:
model.eval()

all_ids = []
all_probs = []

with torch.no_grad():

    for x, r, image_ids in test_loader:

        x = x.to(device, non_blocking=True)
        r = r.to(device, non_blocking=True)

        logits = model(x, r)

        probs = torch.sigmoid(logits)

        all_probs.extend(
            probs.cpu().numpy().tolist()
        )

        all_ids.extend(image_ids)

## 9. Submission

Для каждого тестового изображения `x` создаётся его 180°-версия.

Модель получает пару:

`(x, rot180(x))`

и выдаёт вероятность того, что первое изображение находится в
перевёрнутой ориентации.

Результат сохраняется в `submission.csv` с двумя колонками:

- `image_id`
- `p_180`

In [89]:
submission = pd.DataFrame({
    "image_id": all_ids,
    "p_180": all_probs
})

submission["p_180"] = submission["p_180"].clip(0.0, 1.0)

submission = submission[
    ["image_id", "p_180"]
]

print(submission.head())
print(submission.shape)
print(submission["p_180"].describe())

assert len(submission) == 20000
assert list(submission.columns) == ["image_id", "p_180"]
assert submission["image_id"].is_unique
assert submission["p_180"].between(0, 1).all()

print("All checks passed.")

submission.to_csv(
    "/content/submission.csv",
    index=False
)

print("Saved:", "/content/submission.csv")

from google.colab import files

files.download("/content/submission.csv")

     image_id     p_180
0  test_00000  0.392189
1  test_00001  0.450208
2  test_00002  0.444307
3  test_00003  0.442337
4  test_00004  0.444991
(20000, 2)
count    20000.000000
mean         0.443710
std          0.134111
min          0.013151
25%          0.357759
50%          0.454827
75%          0.531725
max          0.943328
Name: p_180, dtype: float64
All checks passed.
Saved: /content/submission.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>